In [14]:
import pandas as pd
import wandb
from tqdm import tqdm

from utils_modelo_reviews.preprocesamiento import read_reviews, clean_text_lemma
from fastopic import FASTopic
from topmost import Preprocess
from nltk.corpus import stopwords
import spacy
import re
from nltk.tokenize import RegexpTokenizer
from sentence_transformers import SentenceTransformer
import nltk
import numpy as np
import torch
nltk.download('punkt_tab')      
nltk.download('wordnet')    
nltk.download('omw-1.4') 
nltk.download('averaged_perceptron_tagger_eng') 


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\imzli\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\imzli\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\imzli\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\imzli\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [15]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

In [16]:
df = read_reviews()

In [17]:
df.head(100)

,appid,is_positive,weight,text,language
1,747660,True,0.500000,nothing,en
3,747660,True,0.500000,"so, i completed the game and got all the achie...",en
4,747660,True,0.500000,its very good and worth the 40 bucks in my opi...,en
8,747660,True,0.500000,i like that its fun and has a lot of gameplay ...,en
10,747660,True,0.500000,holds a special place in my heart,en
...,...,...,...,...,...
143,747660,True,0.500000,fun and scary,en
146,747660,False,0.527076,"jst not good dont buy/gen, just watch someone ...",en
147,747660,True,0.500000,great game! i have 30 hours of game play on it...,en
148,747660,True,0.500000,the best game ever super fun and very worth it!!!,en


In [18]:
tqdm.pandas(desc="Limpiando texto")

In [19]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

In [21]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "senter", "morphologizer"])

def fast_noun_extractor(texts):
    cleaned = []
    
    for doc in tqdm(nlp.pipe(texts, n_process=4, batch_size=2000), total=len(texts)):
        tokens = [
            t.lemma_.lower() 
            for t in doc 
            if t.pos_ in ("NOUN", "ADJ") and t.is_alpha
        ]
        cleaned.append(" ".join(tokens))
    return cleaned

df["text_clean"] = fast_noun_extractor(df["text"].astype(str).tolist())
df = df[df["text_clean"].str.split().str.len() >= 3].copy()

100%|██████████| 205883/205883 [07:29<00:00, 458.30it/s] 


In [43]:
preprocess = Preprocess(vocab_size=2000, min_doc_count=950, max_doc_freq=0.2)
model = FASTopic(num_topics = 7, verbose= True, preprocess=preprocess,device="cuda", low_memory=True, low_memory_batch_size=50000)

2026-04-07 21:07:07,710 - FASTopic - use device: cuda


In [23]:
def remove_hearts(text):
    text = re.sub(r'(hearts){2,}', '', text)
    text = re.sub('heartsheart', '', text)
    text = re.sub('heart', '', text)
    return text
df["text_clean"] = df["text_clean"].progress_apply(remove_hearts)

Limpiando texto: 100%|██████████| 171631/171631 [00:01<00:00, 161879.38it/s]


In [24]:
texts = df["text"].astype(str).tolist()
model_st = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model_st.encode(texts, show_progress_bar=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1071.40it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 5364/5364 [01:20<00:00, 66.80it/s] 


In [44]:
topic_top_words, doc_topic_dist = model.fit_transform(df["text_clean"].to_list(), preset_doc_embeddings=embeddings)

2026-04-07 21:07:12,177 - FASTopic - Using low memory mode.
2026-04-07 21:07:12,178 - FASTopic - First fit the model.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 952.59it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
parsing texts: 100%|██████████| 171631/171631 [00:04<00:00, 35226.95it/s]
c:\Users\imzli\Documents\Proyecto de Datos\c2526-R4\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-04-07 21:07:29,382 - TopMost - Real vocab size: 712
2026-04-07 21:07:29,391 - TopMost - Real training size: 171631 	 avg len

Topic 0: great lot thing hour bad level way bit gameplay little enemy worth easy new boss
Topic 1: friend mode card control battle single screen team option play deck base war minute match
Topic 2: friend mode card control battle single screen team option play deck minute base able war
Topic 3: steam free developer money year review bug devs update version people issue dead gamer content
Topic 4: amazing cute beautiful room original love perfect horror favorite life awesome funny man simulator girl
Topic 5: amazing cute beautiful room love original perfect horror favorite life awesome funny man girl simulator
Topic 6: story character player experience world design visual art puzzle gameplay mechanic scene choice style narrative


In [ ]:
df["Topic_ID"] = np.argmax(doc_topic_dist, axis=1)

topic_tags = {
    0: "Steam and Dev Support",
    1: "Storytelling",
    2: "Game themes",
    3: "Gameplay mechanics",
    4: "Art and game experience",
}

df["Topic_Name"] = df["Topic_ID"].map(topic_tags)

print(df["Topic_Name"].value_counts())

Topic_Name
Combat & Difficulty    37707
Community & Updates    36977
Arcade & Shooters      28518
Story & Characters     27354
Art & Soundtrack       23545
Price & Value          19266
Design & Immersion     18795
Controls & UI          13721
Name: count, dtype: int64


In [ ]:
pd.set_option('display.max_colwidth', 500)
df

,appid,is_positive,weight,text,language,text_clean,Topic_ID,Topic_Name
1,747660,True,0.5,nothing,en,nothing,9,NaN
3,747660,True,0.5,"so, i completed the game and got all the achievements 100%, even though i'm a fan of fnaf, i didn't think i'd complete it in 3 days.",en,completed game got achievement even though im fan fnaf didnt think id complete day,12,NaN
4,747660,True,0.5,"its very good and worth the 40 bucks in my opinion. warehouse is hard, but beatable, and the maze in mazercise is like impossible. had to watch vid. oerall i love this",en,good worth buck opinion warehouse hard beatable maze mazercise like impossible watch vid oerall love,10,NaN
8,747660,True,0.5,i like that its fun and has a lot of gameplay and content i dont like that if you dont have enough video card memory it crashes when you open certain cams and is laggy,en,like fun lot gameplay content dont like dont enough video card memory crash open certain cam laggy,6,Community & Updates
10,747660,True,0.5,holds a special place in my heart,en,hold special place,9,NaN
...,...,...,...,...,...,...,...,...
261267,3300800,True,0.5,insanely clever puzzles. relaxing music and visuals. highly recommended.,en,insanely clever puzzle relaxing music visuals highly recommended,10,NaN
261268,3300800,True,0.5,good puzzle game with mastercraft design,en,good puzzle game mastercraft design,5,Controls & UI
261269,3300800,True,0.5,"great game, small developer, cheap game, but really enjoyable and probably one of the better ""modern"" puzzle games. highly recommend.",en,great game small developer cheap game really enjoyable probably one better modern puzzle game highly recommend,5,Controls & UI
261270,3300800,True,0.5,"love this game! 10/10, great puzzles and overall design, soundscape etc. well done!",en,love game great puzzle overall design soundscape etc well done,5,Controls & UI


In [ ]:
doc_topic_dist

array([[0.04735397, 0.08469586, 0.05473195, ..., 0.05616326, 0.05465131,
        0.0650663 ],
       [0.0416901 , 0.08791307, 0.05534241, ..., 0.09666675, 0.05223378,
        0.07892337],
       [0.06840822, 0.06446331, 0.06955808, ..., 0.05313357, 0.06029738,
        0.06281132],
       ...,
       [0.07630821, 0.04888268, 0.06779668, ..., 0.06041615, 0.06536679,
        0.0497377 ],
       [0.08563158, 0.05385505, 0.06745337, ..., 0.05105047, 0.05112587,
        0.0436639 ],
       [0.09267574, 0.04585287, 0.07012168, ..., 0.05344379, 0.04572194,
        0.06344092]], shape=(205883, 15), dtype=float32)

In [ ]:
model.get_topic(topic_idx=4)

(('scene', np.float32(0.0269201)),
 ('ending', np.float32(0.021384927)),
 ('voice', np.float32(0.01716365)),
 ('dialogue', np.float32(0.016306672)),
 ('girl', np.float32(0.015573319)))

In [ ]:
fig = model.visualize_topic(top_n=10)
fig.show()

In [ ]:
fig = model.visualize_topic_hierarchy()
fig.show()

c:\Users\imzli\Documents\Proyecto de Datos\c2526-R4\.venv\Lib\site-packages\fastopic\_plot.py:265: ClusterWarning:

The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix



In [ ]:
fig = model.visualize_topic_weights(top_n=20, height=500)
fig.show()